In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm

In [4]:
basepath = GroupPathX('mct-defect')
workpath = basepath['workflows_pbe']


In [6]:
hg = read('Hg.cif')

upd = VaspRelaxUpdater().apply_preset(orm.StructureData(ase=hg), code='vasp-6.3.2@sugon-xh-v2',
                                     overrides={'gga': 'pe'})
upd.set_resources(tot_num_mpiprocs=8, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label('Hg Elemental')
upd.set_relax_settings(algo='rd')
running = upd.submit()
workpath.add_node(running, 'hg_elemental', True)

## Compute Cd

In [7]:
hgte = read('Cd.cif')
upd = VaspRelaxUpdater().apply_preset(orm.StructureData(ase=hgte), 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=16, num_machines=1)
upd.set_options(max_wallclock_seconds=3600 * 2, queue_name='xhhctdnormal')
upd.set_label('Cd RELAX')

upd.builder
running = upd.submit()
workpath.add_node(running, 'cd_elemental_kspacing', True)

## Compute HgTe

In [8]:
hgte = read('HgTe.cif')
upd = VaspRelaxUpdater().apply_preset(orm.StructureData(ase=hgte), 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label('HgTe RELAX')
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte_primitive', True)

## Analyse results

In [15]:
workpath['hgte_primitive'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-14.60104056

In [16]:
workpath['cd_elemental_kspacing'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-1.48850017

In [20]:
workpath['hg_elemental'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-0.53683908

In [26]:
workpath.show_tree(decorate_with_exit_status)

workflows_pbe
├── cd_elemental_kspacing [0]
├── hg_elemental [0]
├── hgte222_V_hg [waiting]
├── hgte333_V_hg [waiting]
└── hgte_primitive [0]



## Proceed with defect calculation

In [30]:
from aiida_user_addons.process.transform import make_vac, make_supercell

In [ ]:
vac_cell = make_vac( workpath['hgte_primitive'].get_node().outputs.relax.structure,[0], [2,2,2])

upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 2, 'ncore':8, 'kpar':2, 'lorbit': None, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 222 V_Hg RELAX')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte222_V_hg')

In [35]:
ref_222 = make_supercell( workpath['hgte_primitive'].get_node().outputs.relax.structure, [2,2,2])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_222, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'ncore':8, 'kpar':2, 
                                                 'lorbit': None, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 222 SUPERCELL RELAX')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte222_supercell')

05/30/2025 09:49:24 PM <2202003> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/30/2025 09:49:24 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662022>: Broadcasting state change: state_changed.created.running
05/30/2025 09:49:24 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662022>: Broadcasting state change: state_changed.running.finished


In [28]:
vac_cell = make_vac( workpath['hgte_primitive'].get_node().outputs.relax.structure,[0], [3,3,3])

upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                      code='vasp-6.3.2-gam@sugon-xh-v2',
                                      overrides={'ispin': 1,
                                                 'ncore':8, 'kpar':1, 'lorbit': None, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 222 V_Hg RELAX')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.set_kpoints_mesh((1,1,1), (0, 0, 0))
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte333_V_hg_gam')

05/30/2025 09:18:01 PM <2202003> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/30/2025 09:18:01 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<661975>: Broadcasting state change: state_changed.created.running
05/30/2025 09:18:01 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<661975>: Broadcasting state change: state_changed.running.finished


In [33]:
ref_333 = make_supercell(workpath['hgte_primitive'].get_node().outputs.relax.structure,[3,3,3])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_333, 
                                      code='vasp-6.3.2-gam@sugon-xh-v2',
                                      overrides={'ispin': 1,
                                                 'ncore':8, 'kpar':1, 'lorbit': None, 'gga': 'pe'}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 333 GAMMMA REF')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.set_kpoints_mesh((1,1,1), (0, 0, 0))
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte333_supercell')

05/30/2025 09:47:50 PM <2202003> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/30/2025 09:47:50 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662001>: Broadcasting state change: state_changed.created.running
05/30/2025 09:47:50 PM <2202003> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662001>: Broadcasting state change: state_changed.running.finished


In [36]:
def read_energy(path):
    return path.get_node().outputs.misc['total_energies']['energy_extrapolated']
def read_energy_per_atom(path):
    node = path.get_node()
    eng = node.outputs.misc['total_energies']['energy_extrapolated']
    return eng / len(node.inputs.structure.sites)

## Compute the formation energy

Using 222 supercell

In [40]:
evac = read_energy(workpath['hgte222_V_hg'])
print(f'Vacancy bearing cell: {evac:.5f} eV')
ebulk = read_energy(workpath['hgte_primitive']) * 8
#ebulk = read_energy(workpath['hgte222_supercell'])
print(f'Bulk cell: {ebulk: .5f} eV')
e_hg = read_energy_per_atom(workpath.browse.hg_elemental())
print(f'Energy per Hg atom: {e_hg: .5f} eV')
e_vac = evac + e_hg - ebulk
print(f'Vacancy formation energy: {e_vac:.5f} eV')

Vacancy bearing cell: -115.25717 eV
Bulk cell: -116.80832 eV
Energy per Hg atom: -0.17895 eV
Vacancy formation energy: 1.37221 eV


Using 333 supercell with Gamma only sampling (216 atoms)

In [45]:
evac = read_energy(workpath['hgte333_V_hg_gam'])
print(f'Vacancy bearing cell: {evac:.5f} eV')
ebulk = read_energy(workpath['hgte333_supercell'])
#ebulk = read_energy(workpath['hgte_primitive']) * 27
print(f'Bulk cell: {ebulk: .5f} eV')
e_hg = read_energy_per_atom(workpath.browse.hg_elemental())
print(f'Energy per Hg atom: {e_hg: .5f} eV')
e_vac = evac + e_hg - ebulk
print(f'Vacancy formation energy: {e_vac:.5f} eV')

Vacancy bearing cell: -392.08159 eV
Bulk cell: -393.40165 eV
Energy per Hg atom: -0.17895 eV
Vacancy formation energy: 1.14111 eV
